# Lesson 5.5 - LLM Post-Training and Fine-Tuning (Transformers + PEFT + TRL)

## Learning Objectives
- Build a **real supervised fine-tuning (SFT)** run scaffold using `transformers`, `peft`, and `trl`.
- Configure and train a **LoRA adapter** with `SFTTrainer`.
- Save/reload adapter artifacts and run inference.
- Use an optional **QLoRA path** (4-bit) when GPU + `bitsandbytes` are available.

This notebook replaces lightweight simulation with concrete trainer flow while remaining small enough for local experimentation.

## 0) Runtime Notes

- This notebook is designed as a **production-style scaffold** with tiny defaults.
- Default model: `sshleifer/tiny-gpt2` so the pipeline is easy to run.
- For meaningful quality, swap in a stronger instruct model and larger curated data.
- Recommended packages:
  - `transformers`, `datasets`, `peft`, `trl`, `accelerate`
  - optional for QLoRA: `bitsandbytes`

In [1]:
from __future__ import annotations

import inspect
import random
from dataclasses import fields
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print({"torch": torch.__version__, "device": device})

{'torch': '2.12.1+cu130', 'device': 'cuda'}


## 1) Create a Small Instruction Dataset

A tiny dataset is enough to validate that the end-to-end trainer stack works.
In real projects, use high-quality, leakage-checked, policy-aligned datasets.

In [2]:
examples = [
    {
        "instruction": "Classify the ticket urgency.",
        "input": "The production checkout API returns 500 for all users.",
        "output": "critical",
    },
    {
        "instruction": "Classify the ticket urgency.",
        "input": "Customer asks for a dark mode option next quarter.",
        "output": "low",
    },
    {
        "instruction": "Extract structured JSON with fields issue_type and priority.",
        "input": "User cannot reset password and sees token expired error repeatedly.",
        "output": '{"issue_type":"auth_reset","priority":"high"}',
    },
    {
        "instruction": "Rewrite for professional support tone.",
        "input": "We messed up your billing, sorry.",
        "output": "We identified a billing issue on our side and are correcting it now.",
    },
    {
        "instruction": "Summarize in one sentence.",
        "input": "Incident timeline: deploy 14:10, errors spike 14:12, rollback 14:18, recovery 14:23.",
        "output": "After deployment, errors spiked and the service recovered following rollback.",
    },
    {
        "instruction": "Classify the ticket urgency.",
        "input": "Weekly report export is delayed by around five minutes.",
        "output": "medium",
    },
    {
        "instruction": "Convert to checklist.",
        "input": "Before launch run integration tests, security scan, and canary rollout.",
        "output": "1. Run integration tests\n2. Run security scan\n3. Execute canary rollout",
    },
    {
        "instruction": "Generate a concise root-cause hypothesis.",
        "input": "Latency increased after enabling synchronous logging in API middleware.",
        "output": "Synchronous logging in request path likely increased blocking I/O and API latency.",
    },
]

raw_ds = Dataset.from_list(examples)
splits = raw_ds.train_test_split(test_size=0.25, seed=SEED)
train_ds, eval_ds = splits["train"], splits["test"]
print({"train_rows": len(train_ds), "eval_rows": len(eval_ds)})

{'train_rows': 6, 'eval_rows': 2}


## 2) Prompt Formatting Function

We keep formatting explicit to avoid silent data-shape errors. This function is passed
into `SFTTrainer(formatting_func=...)`.

In [3]:
def format_example(ex: dict) -> str:
    return (
        "### Instruction:\n"
        f"{ex['instruction']}\n\n"
        "### Input:\n"
        f"{ex['input']}\n\n"
        "### Response:\n"
        f"{ex['output']}"
    )

print(format_example(train_ds[0]))


### Instruction:
Extract structured JSON with fields issue_type and priority.

### Input:
User cannot reset password and sees token expired error repeatedly.

### Response:
{"issue_type":"auth_reset","priority":"high"}


## 3) Load Base Model + Tokenizer

For local scaffolding we use a tiny causal LM. In real fine-tuning, switch to a stronger
instruct-tuned base model compatible with your license and hardware budget.

In [4]:
MODEL_ID = "sshleifer/tiny-gpt2"
MAX_SEQ_LEN = 256


# tokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# model
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
model.config.use_cache = False
print({"model": MODEL_ID, "params": sum(p.numel() for p in model.parameters())})

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'model': 'sshleifer/tiny-gpt2', 'params': 102714}


## 4) Configure LoRA + SFTTrainer

This is the core training scaffold requested:
- `LoraConfig` from PEFT
- `SFTConfig` and `SFTTrainer` from TRL
- concrete trainer construction (not a pseudo-cell)

In [5]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Build SFTConfig with compatibility across TRL/Transformers versions.
sft_fields = {f.name for f in fields(SFTConfig)}
cfg_kwargs = {
    "output_dir": "artifacts/lesson_5_5_lora_sft",
    "num_train_epochs": 1,
    "learning_rate": 2e-4,  # LoRA commonly uses higher LR than full FT
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 2,
    "logging_steps": 1,
    "save_steps": 25,
    "report_to": [],
    "seed": SEED,
}

if "max_length" in sft_fields:
    cfg_kwargs["max_length"] = MAX_SEQ_LEN
if "max_seq_length" in sft_fields:
    cfg_kwargs["max_seq_length"] = MAX_SEQ_LEN
if "eval_strategy" in sft_fields:
    cfg_kwargs["eval_strategy"] = "steps"
elif "evaluation_strategy" in sft_fields:
    cfg_kwargs["evaluation_strategy"] = "steps"
if "eval_steps" in sft_fields:
    cfg_kwargs["eval_steps"] = 1

training_args = SFTConfig(**cfg_kwargs)

trainer_sig = inspect.signature(SFTTrainer.__init__).parameters
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_ds,
    "eval_dataset": eval_ds,
    "peft_config": peft_config,
    "formatting_func": format_example,
}

if "processing_class" in trainer_sig:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_sig:
    trainer_kwargs["tokenizer"] = tokenizer

if "max_seq_length" in trainer_sig:
    trainer_kwargs["max_seq_length"] = MAX_SEQ_LEN
if "dataset_text_field" in trainer_sig:
    # If TRL expects preformatted text field, map one quickly.
    train_fmt = train_ds.map(lambda ex: {"text": format_example(ex)})
    eval_fmt = eval_ds.map(lambda ex: {"text": format_example(ex)})
    trainer_kwargs["train_dataset"] = train_fmt
    trainer_kwargs["eval_dataset"] = eval_fmt
    trainer_kwargs["dataset_text_field"] = "text"
    trainer_kwargs.pop("formatting_func", None)

trainer = SFTTrainer(**trainer_kwargs)

if hasattr(trainer.model, "print_trainable_parameters"):
    trainer.model.print_trainable_parameters()
else:
    trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in trainer.model.parameters())
    print({"trainable": trainable, "total": total, "pct": round(100 * trainable / total, 4)})

/home/ahmad/AI/Github/ai-engineering-zero-to-mastery/.venv/lib/python3.14/site-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Applying formatting function to train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

trainable params: 128 || all params: 102,842 || trainable%: 0.1245


## 5) Run Training + Evaluation

This is a real `trainer.train()` call. Keep epochs small for smoke testing.

In [6]:
train_result = trainer.train()
metrics = trainer.evaluate()
print("train_metrics:", train_result.metrics)
print("eval_metrics:", metrics)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,10.580297,10.625337,10.824578,177.000000,0.000000
2,10.583275,10.625336,10.824578,266.000000,0.000000


Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
10.583275,10.625336,2,10.824578,266.000000,0.000000


train_metrics: {'train_runtime': 1.4098, 'train_samples_per_second': 4.256, 'train_steps_per_second': 1.419, 'total_flos': 547680.0, 'train_loss': 10.581785678863525, 'epoch': 1.0}
eval_metrics: {'eval_loss': 10.625335693359375, 'eval_entropy': 10.824578285217285, 'eval_num_tokens': 266.0, 'eval_mean_token_accuracy': 0.0}


## 6) Save Adapter Artifacts and Test Inference

We save the adapter and tokenizer, then run a quick generation check.

In [7]:
out_dir = Path("artifacts/lesson_5_5_lora_sft/final")
out_dir.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(out_dir)
tokenizer.save_pretrained(out_dir)
print(f"Saved adapter artifacts to: {out_dir}")

prompt = (
    "### Instruction:\n"
    "Classify the ticket urgency.\n\n"
    "### Input:\n"
    "User payments fail across all regions after the latest release.\n\n"
    "### Response:\n"
)

input_tensors = tokenizer(prompt, return_tensors="pt")
input_tensors = {k: v.to(trainer.model.device) for k, v in input_tensors.items()}

with torch.no_grad():
    generated = trainer.model.generate(
        **input_tensors,
        max_new_tokens=32,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(generated[0], skip_special_tokens=True))


Saved adapter artifacts to: artifacts/lesson_5_5_lora_sft/final
### Instruction:
Classify the ticket urgency.

### Input:
User payments fail across all regions after the latest release.

### Response:
 factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors


## 7) Optional QLoRA Path (GPU + bitsandbytes)

This cell provides a concrete QLoRA scaffold. It is optional and hardware-gated.

In [8]:
ENABLE_QLORA_DEMO = False  # flip to True only when CUDA + bitsandbytes are ready

if ENABLE_QLORA_DEMO:
    import torch
    from transformers import BitsAndBytesConfig

    if not torch.cuda.is_available():
        raise RuntimeError("QLoRA demo requires CUDA.")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    q_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )

    q_peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )

    q_cfg_kwargs = dict(cfg_kwargs)
    q_cfg_kwargs["output_dir"] = "artifacts/lesson_5_5_qlora_sft"
    q_training_args = SFTConfig(**q_cfg_kwargs)

    q_trainer_kwargs = {
        "model": q_model,
        "args": q_training_args,
        "train_dataset": train_ds,
        "eval_dataset": eval_ds,
        "peft_config": q_peft_config,
        "formatting_func": format_example,
    }
    if "processing_class" in trainer_sig:
        q_trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_sig:
        q_trainer_kwargs["tokenizer"] = tokenizer

    q_trainer = SFTTrainer(**q_trainer_kwargs)
    q_train_result = q_trainer.train()
    print("QLoRA train metrics:", q_train_result.metrics)
else:
    print("Skipped QLoRA path. Set ENABLE_QLORA_DEMO=True to run.")

Skipped QLoRA path. Set ENABLE_QLORA_DEMO=True to run.


## Capstone-Style Checklist for Real Projects

- Replace toy dataset with versioned production-style data.
- Add strict train/validation/test split with leakage checks.
- Add evaluation suite (task quality, format adherence, safety, latency, cost).
- Register artifacts and metadata in MLflow/W&B.
- Deploy adapter with canary/shadow rollout and rollback switch.

## Case Studies & Exceptions

1. **Customer support assistant**
   - Use LoRA SFT to enforce response format and support policy tone.
   - Exception: if failures are mostly factual freshness, prioritize RAG/index refresh over more tuning.

2. **Internal engineering copilot**
   - Use PEFT to align patch style and command safety with team conventions.
   - Exception: if repository context retrieval is weak, tuning alone will not fix hallucinated paths.

3. **Resource-constrained startup**
   - Use QLoRA to train adapters on limited hardware.
   - Exception: for strict high-precision domains, quantized training may require heavier validation before release.

## Interview Questions & Answers

1. **Q: Why use PEFT instead of full fine-tuning?**  
   **A:** PEFT reduces memory/compute cost by training a small adapter subset while keeping most base weights frozen.

2. **Q: What is LoRA in one line?**  
   **A:** LoRA learns low-rank weight updates injected into selected layers of a frozen model.

3. **Q: What is QLoRA?**  
   **A:** QLoRA combines 4-bit quantized base weights with LoRA adapters to cut memory usage further.

4. **Q: Why does LoRA often use higher learning rates?**  
   **A:** Fewer trainable parameters usually need larger updates to converge effectively.

5. **Q: What does `SFTTrainer` handle for you?**  
   **A:** Dataset formatting/tokenization pipeline, trainer wiring, and PEFT integration with standard training loops.

6. **Q: When is fine-tuning the wrong first move?**  
   **A:** When errors come from stale knowledge; use RAG/data refresh first.

7. **Q: What should you log for reproducibility?**  
   **A:** Base model revision, dataset version hash, hyperparameters, eval results, and adapter artifact IDs.

8. **Q: How do you avoid leakage in post-training?**  
   **A:** Split data before curation, deduplicate across splits, and keep evaluation prompts isolated.

9. **Q: Can tuned adapters replace runtime guardrails?**  
   **A:** No. Guardrails and policy checks remain necessary defense-in-depth controls.

10. **Q: What is a practical release strategy?**  
    **A:** Shadow/canary release with quality and safety gates, then gradual traffic ramp-up.

11. **Q: What is the main business benefit of this workflow?**  
    **A:** More consistent task behavior at lower adaptation cost than full model retraining.

12. **Q: What is a common anti-pattern in fine-tuning projects?**  
    **A:** Spending time on hyperparameter tweaks before building a credible eval harness.